# CAG: Cache-Augmented Generation

**Idea:** the knowledge is small enough to fit in the prompt. So put **all of it** in the
prompt once, let the API **cache** it, and reuse the cached copy for every question.

No cleaning, no chunking, no embeddings, no vector database, no search.

```
all documents ──► one big prompt (cached) ──► question 1, question 2, question 3 ...
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent            # the demos/ folder
DATA = ROOT / "data"                # the knowledge: the same files for all three demos
assert DATA.exists(), "Open this notebook from demos/notebooks/"
sys.path.append(str(ROOT))
import importlib, llm
importlib.reload(llm)               # picks up edits to llm.py when this cell is re-run
from llm import ask                 # small helper that calls Claude and shows the chat

## 1. Load every document, as it is

In [ ]:
import pandas as pd

docs = [p for p in sorted(DATA.iterdir()) if p.suffix in (".html", ".txt", ".md")]
pd.DataFrame({"file": [p.name for p in docs], "characters": [p.stat().st_size for p in docs]})

## 2. Put them all in the system prompt

In [ ]:
knowledge = "\n\n".join(f"<document name='{p.name}'>\n{p.read_text(encoding='utf-8')}</document>"
                         for p in docs)
system = "Answer the employee's question using only these company documents.\n\n" + knowledge
print(f"{len(system):,} characters in the prompt (about {len(system) // 4:,} tokens)")
print(system[:500], "...")

## 3. First question: the documents are processed and **written** to the cache

Look at the `Tokens:` line under the answer: `cache_write` is large.

In [ ]:
ask(system, "Can I roll over unused vacation days?", cache=True);

## 4. Next questions: the documents are **read** from the cache

Same prompt prefix, so `cache_read` is large now: cheaper and faster.

In [ ]:
ask(system, "Can I expense a business-class flight to London?", cache=True);

Even a question that needs facts from **two documents** works, because every fact is
in the prompt: the carry-over rule (leave policy) and Omar's manager (team directory).

In [ ]:
ask(system, "Who approves Omar's leave carry-over?", cache=True);

## Try your own question

In [ ]:
ask(system, "Is parking free?", cache=True);

## Takeaways

- ✅ Simplest possible setup, and the model sees everything, so connected questions work.
- ✅ Caching makes repeated questions cheap and fast.
- ❌ Only works while the knowledge **fits** in the context window.
- ❌ Every question pays for reading all of it (even from cache), and quality can drop with very long prompts.